# 004 OpenAI Function Calling

这是第四份 OpenAI 学习 Notebook。

学习目标：

1. 理解 Function Calling 为什么是 Agent 的核心能力之一
2. 学会定义工具 `tools`
3. 学会让模型决定调用哪个函数
4. 学会在应用侧真正执行本地函数，并把结果回传给模型

这一阶段的关键变化是：

- 前三份 Notebook 里，模型主要负责“说”
- 从这一份开始，模型需要“决定调用你的函数”

参考文档：

- OpenAI Function Calling：https://platform.openai.com/docs/guides/function-calling


## 先理解核心概念

Function Calling 不是“模型直接执行代码”。

真实过程通常分成两步：

1. 模型先告诉你“它想调用哪个工具，以及参数是什么”
2. 你的应用代码真正执行本地函数，再把执行结果回传给模型

你可以把它理解成：

- 模型：负责做“决策”
- 应用程序：负责做“执行”

这和 Java 里的 Controller / Service 分工有点像：

- 模型更像一个会决定“该调哪个 Service 方法”的智能路由层
- 你的 Python 代码仍然是最终执行方


## 加载环境变量

这里继续沿用前几份 Notebook 的方式：自动向上查找项目根目录 `.env`。


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)


Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 创建客户端

这份 Notebook 继续使用 `Chat Completions API`，因为很多私有兼容网关都优先支持这条路径。


In [2]:
from openai import OpenAI


if not OPENAI_API_KEY:
    raise ValueError("请先在项目根目录 .env 中配置 OPENAI_API_KEY")


client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


## 先写几个本地 mock 工具

这一阶段先不要急着接真实股票 API。

先用固定返回值把链路跑通，重点理解：

- 模型如何选工具
- 应用如何执行工具
- 工具结果如何再交回模型

这几个函数就是你应用侧真正会执行的本地代码。


In [3]:
def get_stock_quote(symbol: str) -> dict:
    mock_data = {
        "AAPL": {"symbol": "AAPL", "price": 215.32, "change_percent": 1.82},
        "NVDA": {"symbol": "NVDA", "price": 118.47, "change_percent": -0.64},
        "TSLA": {"symbol": "TSLA", "price": 177.91, "change_percent": 2.15},
    }
    return mock_data.get(symbol.upper(), {"symbol": symbol.upper(), "price": None, "change_percent": None})


def get_stock_news(symbol: str) -> dict:
    mock_news = {
        "AAPL": [
            "苹果继续推进生成式 AI 能力整合。",
            "市场关注新一季 iPhone 销售预期。",
        ],
        "NVDA": [
            "英伟达数据中心业务仍然是市场焦点。",
            "AI 芯片供应链动态持续受关注。",
        ],
    }
    return {"symbol": symbol.upper(), "news": mock_news.get(symbol.upper(), ["暂无新闻数据"]) }


def get_company_profile(symbol: str) -> dict:
    mock_profiles = {
        "AAPL": {"symbol": "AAPL", "company_name": "Apple Inc.", "industry": "Consumer Electronics"},
        "NVDA": {"symbol": "NVDA", "company_name": "NVIDIA Corporation", "industry": "Semiconductors"},
    }
    return mock_profiles.get(symbol.upper(), {"symbol": symbol.upper(), "company_name": "Unknown", "industry": "Unknown"})


## 定义工具描述

这里的 `tools` 不是 Python 函数本身，而是“给模型看的工具说明书”。

模型会根据这些描述判断：

- 有哪些工具可用
- 每个工具是干什么的
- 调用时需要什么参数

这有点像给模型一份“可调用接口的 OpenAPI 摘要”。


In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_quote",
            "description": "查询股票当前价格和涨跌幅",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "股票代码，例如 AAPL、NVDA、TSLA",
                    }
                },
                "required": ["symbol"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_news",
            "description": "查询某只股票最近的新闻摘要",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "股票代码，例如 AAPL、NVDA、TSLA",
                    }
                },
                "required": ["symbol"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_company_profile",
            "description": "查询公司的基本信息，例如公司名称和行业",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol": {
                        "type": "string",
                        "description": "股票代码，例如 AAPL、NVDA、TSLA",
                    }
                },
                "required": ["symbol"],
                "additionalProperties": False,
            },
        },
    },
]

tools


[{'type': 'function',
  'function': {'name': 'get_stock_quote',
   'description': '查询股票当前价格和涨跌幅',
   'parameters': {'type': 'object',
    'properties': {'symbol': {'type': 'string',
      'description': '股票代码，例如 AAPL、NVDA、TSLA'}},
    'required': ['symbol'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_stock_news',
   'description': '查询某只股票最近的新闻摘要',
   'parameters': {'type': 'object',
    'properties': {'symbol': {'type': 'string',
      'description': '股票代码，例如 AAPL、NVDA、TSLA'}},
    'required': ['symbol'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_company_profile',
   'description': '查询公司的基本信息，例如公司名称和行业',
   'parameters': {'type': 'object',
    'properties': {'symbol': {'type': 'string',
      'description': '股票代码，例如 AAPL、NVDA、TSLA'}},
    'required': ['symbol'],
    'additionalProperties': False}}}]

## 准备系统提示词

提示词的重点不是让模型直接给答案，而是：

- 优先判断是否需要调用工具
- 能调工具时，就不要硬编数据
- 拿到工具结果后，再组织自然语言回答


In [5]:
SYSTEM_PROMPT = """
你是一个股票助手。

你的工作规则：
1. 如果用户在问股价、涨跌幅、公司信息、相关新闻，优先调用工具获取数据。
2. 不要编造股票价格、新闻或公司资料。
3. 如果用户问题不明确，可以先澄清。
4. 在拿到工具结果后，用简洁中文给出结论。
""".strip()


## 先发起第一轮请求，让模型决定是否调用工具

这里先只把用户问题和 `tools` 发给模型。

如果模型觉得需要调用工具，返回结果里通常会出现：

- `finish_reason = "tool_calls"`
- `message.tool_calls`

也就是说，模型不是直接回答，而是先提出“请你帮我调用这个函数”。


In [6]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "苹果现在股价多少？"},
]

first_response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=messages,
    tools=tools,
)

first_choice = first_response.choices[0]
print("finish_reason =", first_choice.finish_reason)
print("tool_calls =", first_choice.message.tool_calls)


finish_reason = tool_calls
tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_d7a3916ff56142f684ac2bc7', function=Function(arguments='{"symbol": "AAPL"}', name='get_stock_quote'), type='function')]


## 看懂 `tool_calls` 的结构

`tool_calls` 里最关键的是三部分：

- `id`：这次工具调用的唯一标识
- `function.name`：模型要调用哪个函数
- `function.arguments`：函数参数，通常是 JSON 字符串

你可以把它理解成：模型已经帮你组装好了一次“方法调用请求”。


In [7]:
import json


tool_call = first_choice.message.tool_calls[0]
print("tool_call_id =", tool_call.id)
print("function_name =", tool_call.function.name)
print("raw_arguments =", tool_call.function.arguments)
print("parsed_arguments =", json.loads(tool_call.function.arguments))


tool_call_id = call_d7a3916ff56142f684ac2bc7
function_name = get_stock_quote
raw_arguments = {"symbol": "AAPL"}
parsed_arguments = {'symbol': 'AAPL'}


## 建立函数分发表

模型只会告诉你函数名字符串，不会自动执行 Python 函数。

所以应用侧通常要准备一个分发表，把：

- `"get_stock_quote"`
- `"get_stock_news"`
- `"get_company_profile"`

映射到真正的 Python 函数对象。

这和 Java 里自己维护一个 `Map<String, Handler>` 很像。


In [8]:
FUNCTION_REGISTRY = {
    "get_stock_quote": get_stock_quote,
    "get_stock_news": get_stock_news,
    "get_company_profile": get_company_profile,
}


## 执行模型请求的工具调用

这一步非常关键。

现在真正执行函数的是你的应用，而不是模型。

执行流程是：

1. 取出 `function.name`
2. 解析 `function.arguments`
3. 找到本地函数
4. 执行函数
5. 把结果转成字符串，准备回传给模型


In [9]:
def run_tool_call(tool_call) -> dict:
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    if function_name not in FUNCTION_REGISTRY:
        raise ValueError(f"未知工具: {function_name}")

    result = FUNCTION_REGISTRY[function_name](**function_args)
    return {
        "tool_call_id": tool_call.id,
        "function_name": function_name,
        "arguments": function_args,
        "result": result,
    }


executed_tool = run_tool_call(tool_call)
executed_tool


{'tool_call_id': 'call_d7a3916ff56142f684ac2bc7',
 'function_name': 'get_stock_quote',
 'arguments': {'symbol': 'AAPL'},
 'result': {'symbol': 'AAPL', 'price': 215.32, 'change_percent': 1.82}}

## 把工具结果回传给模型

到这里还没结束。

工具执行完之后，模型只拿到了“它曾经请求过一个工具”，但还不知道工具返回了什么。

所以第二轮请求要把三类消息一起发回去：

1. 原始对话消息
2. 模型刚才发出的 assistant tool call 消息
3. 你作为应用补上的 tool 结果消息

注意这里 `role` 是 `tool`，并且要带上 `tool_call_id`，这样模型才能把结果和之前那次调用对应起来。


In [10]:
messages.append(first_choice.message)
messages.append(
    {
        "role": "tool",
        "tool_call_id": executed_tool["tool_call_id"],
        "content": json.dumps(executed_tool["result"], ensure_ascii=False),
    }
)

messages


[{'role': 'system',
  'content': '你是一个股票助手。\n\n你的工作规则：\n1. 如果用户在问股价、涨跌幅、公司信息、相关新闻，优先调用工具获取数据。\n2. 不要编造股票价格、新闻或公司资料。\n3. 如果用户问题不明确，可以先澄清。\n4. 在拿到工具结果后，用简洁中文给出结论。'},
 {'role': 'user', 'content': '苹果现在股价多少？'},
 ChatCompletionMessage(content='\n\n', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d7a3916ff56142f684ac2bc7', function=Function(arguments='{"symbol": "AAPL"}', name='get_stock_quote'), type='function')], reasoning_content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "苹果现在股价多少？" (What is Apple\'s current stock price?)\n   - Key entity: 苹果 (Apple)\n   - Required information: Current stock price (and likely symbol)\n   - Apple\'s stock symbol is typically "AAPL".\n\n2.  **Identify Required Tool:**\n   - Need to get stock quote (price, change, etc.)\n   - Tool: `get_stock_quote`\n   - Parameter: `symbol` = "AAPL"\n\n3.  **Execute Tool Call:**\n   - Call `get_s

## 发起第二轮请求，让模型组织最终答案

这一次模型已经拿到了真实工具结果。

所以它不需要再决定调用哪个工具，而是应该基于工具结果，整理成人类可读的自然语言回答。


In [11]:
second_response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=messages,
    tools=tools,
)

final_answer = second_response.choices[0].message.content or ""
print(final_answer)




苹果公司（AAPL）当前股价为 **215.32 美元**，今日上涨约 **1.82%**。（数据为实时行情，仅供参考）


## 封装成一个完整函数

前面我们是拆开演示每一步。

真实项目里，通常会把这套流程封装成一个服务方法，比如：

- 接收用户消息
- 发第一轮请求
- 执行工具
- 回传工具结果
- 返回最终答案

这就是最小的“工具调用 Agent 回合”。


In [12]:
def chat_with_tools(user_message: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    first_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        tools=tools,
    )
    first_choice = first_response.choices[0]

    tool_calls = first_choice.message.tool_calls or []
    if not tool_calls:
        return first_choice.message.content or ""

    messages.append(first_choice.message)

    for tool_call in tool_calls:
        executed = run_tool_call(tool_call)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": executed["tool_call_id"],
                "content": json.dumps(executed["result"], ensure_ascii=False),
            }
        )

    second_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        tools=tools,
    )
    return second_response.choices[0].message.content or ""


## 试几个问题

现在你可以用自然语言提问，模型会自己判断要不要调用工具。


In [13]:
questions = [
    "苹果现在股价多少？",
    "英伟达最近有什么新闻？",
    "AAPL 是什么公司？",
]

for question in questions:
    print("user:", question)
    print("assistant:", chat_with_tools(question))
    print("-" * 60)


user: 苹果现在股价多少？
assistant: 

苹果公司（AAPL）当前股价为 **215.32美元**，今日涨幅为 **1.82%**。
------------------------------------------------------------
user: 英伟达最近有什么新闻？
assistant: 

英伟达（NVDA）最近的新闻摘要如下：

1. 英伟达数据中心业务仍然是市场焦点。
2. AI 芯片供应链动态持续受关注。
------------------------------------------------------------
user: AAPL 是什么公司？
assistant: 

AAPL 是 **Apple Inc.**（苹果公司）的股票代码，该公司主要属于 **消费电子（Consumer Electronics）** 行业。
------------------------------------------------------------


## 这一步和 Structured Outputs 的关系

你可以这样理解两者分工：

- Structured Outputs：让模型稳定输出“结构化字段”
- Function Calling：让模型稳定决定“该调用哪个工具”

真实 Agent 往往会把两者串起来：

1. 先做意图识别
2. 再做工具选择
3. 最后组织自然语言答案


## 为什么 Function Calling 比纯 Prompt 更可靠

如果你只靠提示词让模型输出：

```python
请返回：get_stock_quote(symbol="AAPL")
```

模型虽然可能看起来像在“调用函数”，但本质上仍然只是生成文本。

Function Calling 的关键价值在于：

- 工具列表是显式给模型的
- 参数结构是显式定义的
- 应用端能明确区分“普通回答”和“工具调用请求”

这会让系统稳定很多。


## 常见坑

1. 模型会决定调用工具，但不会帮你执行本地代码。
2. 工具参数是字符串 JSON，应用端需要自己 `json.loads(...)`。
3. 工具结果回传时，`tool_call_id` 必须对应上。
4. 有些私有兼容网关对 `tools` 支持不完整，必要时要做兼容测试。
5. 第一版不要接太多工具，否则你很难判断是提示词问题、工具定义问题，还是执行流程问题。


## 当前阶段结论

你现在需要记住：

1. Function Calling 的本质是“模型决定调用什么，应用真正执行什么”
2. `tools` 是给模型看的工具说明，不是函数本体
3. `tool_calls` 表示模型想调用工具
4. 你的代码必须执行工具，再把结果作为 `role=tool` 消息回传
5. 这一套跑通之后，才算真正具备了最小 Agent 能力

下一份建议学习：

- 做一个最小股票智能体 Demo
- 再把 Structured Outputs 和 Function Calling 串起来
